# 09 — Computer vision with a convolutional network

**Level:** Intermediate · **Study time:** 75–100 minutes (including experiments)

**Prerequisites:** Notebooks 06–08.

## What you will be able to do
- Train on real handwritten digit images.
- Trace convolution and pooling shapes.
- Inspect held-out mistakes, not just average accuracy.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 08](08_generalization.ipynb) · [Next: 10](10_transfer_learning.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Load a small real dataset without downloads

Scikit-learn bundles 1,797 handwritten 8 × 8 digit images from the UCI optical recognition dataset. Pixel intensities range from 0 to 16. This fixed known range lets us scale by 16 without estimating statistics from validation data. Stratification preserves approximate class proportions.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
X = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
y = torch.tensor(digits.target, dtype=torch.long)
tr, va = train_test_split(np.arange(len(y)), test_size=0.25, random_state=SEED, stratify=y.numpy())
loader = DataLoader(TensorDataset(X[tr], y[tr]), batch_size=64, shuffle=True, num_workers=0)
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(X[i, 0], cmap="gray"); ax.set_title(str(y[i].item())); ax.axis("off")
plt.show()
print("[batch, channels, height, width]:", X.shape)


## Step 3 — Build local pattern detectors

A convolution reuses a small filter across locations. Padding 1 with a 3 × 3 kernel preserves spatial size at stride 1. Pooling reduces 8 × 8 to 4 × 4. The final linear layer consumes the flattened feature maps.


In [ ]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
                                      nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * 4 * 4, 10))
    def forward(self, x):
        return self.head(self.features(x))

model = DigitCNN()
probe = X[:4]
for layer in model.features:
    probe = layer(probe)
    print(type(layer).__name__, tuple(probe.shape))
assert model(X[:4]).shape == (4, 10)


## Step 4 — Train and track validation accuracy

Use raw logits with cross entropy. This is a development experiment: the held-out split is validation because we inspect it during training. The capstone adds a separate final test set.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
history = []
for epoch in range(15):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(xb), yb)
        loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        accuracy = (model(X[va]).argmax(1) == y[va]).float().mean().item()
    history.append(accuracy)
print("Validation accuracy:", history[-1])
assert history[-1] > 0.85
plt.plot(range(1, len(history) + 1), history); plt.xlabel("Epoch"); plt.ylabel("Validation accuracy"); plt.show()


## Step 5 — Inspect mistakes

A confusion matrix reveals which classes are confused. Looking at individual errors can suggest problems with data quality or model assumptions. Do not assume that confidence guarantees correctness.


In [ ]:
with torch.no_grad():
    probs = model(X[va]).softmax(1)
pred = probs.argmax(1)
confusion = torch.bincount(y[va] * 10 + pred, minlength=100).reshape(10, 10)
plt.imshow(confusion, cmap="Blues"); plt.xlabel("Predicted"); plt.ylabel("True"); plt.colorbar(); plt.show()
errors = (pred != y[va]).nonzero().flatten()[:6]
if len(errors):
    fig, axes = plt.subplots(1, len(errors), figsize=(12, 2), squeeze=False)
    for ax, i in zip(axes[0], errors.tolist()):
        ax.imshow(X[va[i], 0], cmap="gray")
        ax.set_title(f"True {y[va[i]].item()} / Pred {pred[i].item()}"); ax.axis("off")
    plt.show()
else:
    print("No validation errors in this run.")


## Your turn — practice before opening the solutions

1. Count the first convolution's parameters by hand.
2. Visualize the first eight learned filters.
3. Compute macro recall (the mean of class recalls) and compare it with accuracy.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
conv = model.features[0]
assert sum(p.numel() for p in conv.parameters()) == 16 * (1 * 3 * 3 + 1)
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(conv.weight[i, 0].detach(), cmap="coolwarm"); ax.axis("off")
plt.show()
macro_recall = (confusion.diag() / confusion.sum(1).clamp_min(1)).mean().item()
print("Macro recall:", macro_recall, "Accuracy:", history[-1])


## Before moving on

You have trained a real image classifier. The small bundled dataset is useful for learning, but it does not establish performance on photos or handwriting from a new source.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [Bundled digits dataset and provenance](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)


[Course guide](../README.md) · [Previous: 08](08_generalization.ipynb) · [Next: 10](10_transfer_learning.ipynb)
